# Генерация текста при помощи LLM

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы: 
* https://huggingface.co/docs/transformers/llm_tutorial
* https://huggingface.co/docs/transformers/v5.3.0/generation_strategies
* https://huggingface.co/docs/transformers/perplexity
* https://machinelearningmastery.com/evaluating-perplexity-on-language-models/


## Задачи для совместного разбора

1\. Обсудите сценарии работы с языковыми моделями из пакета `transformers` для задачи генерации текста.

## Задачи для самостоятельного решения

In [26]:
import math
import random

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    PreTrainedModel,
    PreTrainedTokenizer,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


<p class="task" id="1"></p>

1\. Загрузите модель `ai-forever/rugpt3small_based_on_gpt2` (или другую небольшую GPT) и соответствующий токенизатор из библиотеки `transformers`.

Реализуйте функцию `generate_manual(prompt, steps=10)`. Напишите цикл вручную:
- Токенизируйте `prompt`.
- Цикл `range(steps)`:
  - подайте текущую последовательность в модель.
  - возьмите логиты последнего токена
  - выберите следующий токен жадно: `next_token = torch.argmax(logits)`.
  - добавьте `next_token` к последовательности.
- После завершения цика декодируйте токены и выведите результат.

Сравните результат с `model.generate(..., do_sample=False)`. 

- [ ] Проверено на семинаре

In [27]:
model_name = "ai-forever/rugpt3small_based_on_gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()


def generate_manual(
    prompt: str, steps: int = 10, model: PreTrainedModel = None, tokenizer: PreTrainedTokenizer = None
) -> str:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    for _ in range(steps):
        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits
            last_token_logits = logits[0, -1, :]
            next_token_id = torch.argmax(last_token_logits, dim=-1)
            next_token_tensor = next_token_id.unsqueeze(0).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_token_tensor], dim=-1)

    generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return generated_text


prompt = "Постой, не уходи. Мы "
steps = 10

print("Ручная генерация")
manual_output = generate_manual(
    prompt, steps=steps, model=model, tokenizer=tokenizer
)
print(manual_output)

print("Встроенная генерация (model.generate)")
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    builtin_output_ids = model.generate(
        input_ids, max_new_tokens=steps, do_sample=False
    )
builtin_output = tokenizer.decode(
    builtin_output_ids[0], skip_special_tokens=True
)
print(builtin_output)

assert (
    manual_output == builtin_output
), "Результаты ручной и встроенной генерации должны совпадать!"
print("\nРезультаты полностью идентичны.")

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ручная генерация
Постой, не уходи. Мы  не можем  уйти  от  тебя  и
Встроенная генерация (model.generate)
Постой, не уходи. Мы  не можем  уйти  от  тебя  и

Результаты полностью идентичны.


<p class="task" id="2"></p>

2\. Используя встроенный метод `model.generate()`, исследуйте, как параметры процесса генерации влияют на текст. Возьмите промпт: "В 2026 году искусственный интеллект научился".

Сгенерируйте продолжение (20 токенов) с разными настройками :
- do_sample;
- num_beams
- temperature

Для каждой конфигурации запустите процесс генерации 3 раза. Выведите на экран результаты. 

Прокомментируйте, какие настройки позволяют получить более осмысленные тексты, какие - более разнообразные. Объясните, почему.

- [ ] Проверено на семинаре


In [28]:
prompt_task2 = "В 2026 году искусственный интеллект научился"
max_new_tokens = 20

configs = [
    {
        "name": "Greedy (Обычный жадный поиск)",
        "params": {"do_sample": False, "num_beams": 1},
    },
    {
        "name": "Beam Search (Поиск по лучам, num_beams=5)",
        "params": {"do_sample": False, "num_beams": 3},
    },
    {
        "name": "Sampling Low Temp (do_sample=True, temp=0.3)",
        "params": {"do_sample": True, "temperature": 0.3, "top_k": 50},
    },
    {
        "name": "Sampling High Temp (do_sample=True, temp=1.5)",
        "params": {"do_sample": True, "temperature": 1.5, "top_k": 50},
    },
    {
        "name": "Sampling Very High Temp (do_sample=True, temp=2)",
        "params": {"do_sample": True, "temperature": 2.0, "top_k": 50},
    },
    {
        "name": "Sampling So high Temp [You can't imagine how high] (do_sample=True, temp=3)",
        "params": {"do_sample": True, "temperature": 3.0, "top_k": 50},
    },
]

input_ids_task2 = tokenizer.encode(prompt_task2, return_tensors="pt").to(
    device
)

for config in configs:
    print(f"Конфигурация: {config['name']}")
    for i in range(3):
        with torch.no_grad():
            gen_ids = model.generate(
                input_ids_task2,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id,
                **config["params"],
            )
        output_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
        print(f"Запуск {i+1}: {output_text}")

Конфигурация: Greedy (Обычный жадный поиск)
Запуск 1: В 2026 году искусственный интеллект научился распознавать и распознавать лица людей.  В 2026 году искусственный интеллект научился распознавать и распозна
Запуск 2: В 2026 году искусственный интеллект научился распознавать и распознавать лица людей.  В 2026 году искусственный интеллект научился распознавать и распозна
Запуск 3: В 2026 году искусственный интеллект научился распознавать и распознавать лица людей.  В 2026 году искусственный интеллект научился распознавать и распозна
Конфигурация: Beam Search (Поиск по лучам, num_beams=5)
Запуск 1: В 2026 году искусственный интеллект научился распознавать лица людей.















Запуск 2: В 2026 году искусственный интеллект научился распознавать лица людей.















Запуск 3: В 2026 году искусственный интеллект научился распознавать лица людей.















Конфигурация: Sampling Low Temp (do_sample=True, temp=0.3)
Запуск 1: В 2026 году искусственный интеллект научился распоз

(я не знаю, что с выводом)

1. **Greedy search (`do_sample=False`)**: Все 3 запуска выдают абсолютно одинаковый текст. Модель выбирает локально оптимальные слова, поэтому текст может казаться зацикленным или слишком банальным.
2. **Beam Search (`num_beams=5`)**: Тексты также одинаковые для всех запусков (т.к. сэмплирование выключено), но они гораздо более связные и "умные", чем при обычном жадном поиске. Модель жертвует сиюминутной выгодой ради общей логики предложения.
3. **Low Temperature (`temp=0.3`)**: Тексты получаются разнообразными от запуска к запуску, но при этом остаются весьма осмысленными и грамматически верными. Это идеальный баланс (компромисс Bias vs Variance) для большинства практических задач.
4. **High Temperature (`temp=1.5`)**: Тексты становятся очень разнообразными, но теряют смысл. Модель начинает генерировать бессвязные наборы слов, пунктуационный хаос и теряет нить повествования.
5. **Very High Temperature (`temp=2`)**: Слова связаны между собой, но смысла нету никакого.
6. **So High Temperature (`temp=3`)**: Начинает казаться, что слова связаны попарно, но общего смысла нету. 

**Вывод:** Настройки `num_beams` и низкая температура дают более осмысленные тексты. Высокая температура и включенное сэмплирование дают разнообразие, но снижают качество.

<p class="task" id="3"></p>

3\. Загрузите токенизатор от любой современной Instruct-модели (например, `Qwen/Qwen2.5-1.5B-Instruct`). Саму модель не нужно загружать, только токенизатор.

Создайте структуру диалога:
```python
messages = [
    {"role": "system", "content": "Ты вежливый помощник."},
    {"role": "user", "content": "Как сварить пельмени?"},
]
```

Примените метод `tokenizer.apply_chat_template(messages, tokenize=False)`. Выведите полученную строку. Прокомментируйте особенности полученного шаблона.

- [ ] Проверено на семинаре


In [29]:
instruct_model_name = "Qwen/Qwen3-4B-Instruct-2507"
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

messages = [
    {"role": "system", "content": "Ты вежливый помощник."},
    {"role": "user", "content": "Как сварить пельмени?"},
]

# Применяем шаблон чата
formatted_chat = instruct_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(formatted_chat)

<|im_start|>system
Ты вежливый помощник.<|im_end|>
<|im_start|>user
Как сварить пельмени?<|im_end|>
<|im_start|>assistant



В полученной строке мы видим специальные служебные токены разметки:
- `<|im_start|>` и `<|im_end|>` — маркеры начала и конца реплики определенной роли.
- После `<|im_start|>` идет указание роли (`system`, `user`, `assistant`), чтобы модель четко разграничивала инструкции, вопросы человека и свои собственные ответы.
- В самом конце шаблона (благодаря аргументу `add_generation_prompt=True`) токенизатор оставил открытый тег `<|im_start|>assistant\n`. Это служит явным сигналом для модели: "Твоя очередь говорить,ты, ничтожная машина для генирации текстов, начни генерировать текст именно отсюда".

<p class="task" id="4"></p>

4\. Возьмите любой осмысленный текст на русском языке длиной 15-20 слов (например, заголовок новости или предложение из книги, лучше без имен собственных). Напишите функцию расчета перплексии для фразы, которая:
- токенизирует текст;
- получает прогнозы модели;
- считает cross-entropy loss (если передать аргумент labels, модель сама посчитает loss);
- на основе CE считает перплексию.

Напишите функцию расчета перплексии для фразы. Загрузите любую компактную языковую модель с головой для языкового моделирования (`AutoModelForCausalLM`).

Посчитайте перплексию для исходной фразы. Перемешайте слова в исходном предложении в случайном порядке и посчитайте перплексию для этой фразы. Выведите оба значения и прокомментируйте результат.

- [ ] Проверено на семинаре


In [30]:
def compute_perplexity(text: str, model: PreTrainedModel, tokenizer: PreTrainedTokenizer) -> float:
    """Вычисляет перплексию (Perplexity) для заданной строки текста."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    inputs["labels"] = inputs["input_ids"].clone()
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    loss = outputs.loss
    perplexity = math.exp(loss.item())
    return perplexity

original_text = "Прочность емкости имела не меньшее значение, чем сила взрывчатки. Если взять хрупкий сосуд, получится огненный шар без особой ударной силы."
# предложение одной из последних страниц книги "Марсианин" (Энди Веир)
print(f"Исходный текст (длина {len(original_text.split())} слов):")
print(original_text)

ppl_original = compute_perplexity(original_text, model, tokenizer)
print(f"Перплексия исходного текста: {ppl_original:.4f}")

words = original_text.split()
random.shuffle(words)
shuffled_text = " ".join(words)

print("\nПеремешанный текст:")
print(shuffled_text)

ppl_shuffled = compute_perplexity(shuffled_text, model, tokenizer)
print(f"Перплексия перемешанного текста: {ppl_shuffled:.4f}")

Исходный текст (длина 20 слов):
Прочность емкости имела не меньшее значение, чем сила взрывчатки. Если взять хрупкий сосуд, получится огненный шар без особой ударной силы.
Перплексия исходного текста: 44.2698

Перемешанный текст:
силы. значение, огненный меньшее Если получится шар ударной чем сосуд, особой взять емкости хрупкий имела без сила взрывчатки. Прочность не
Перплексия перемешанного текста: 2190.1091


Как и ожидалось, перплексия перемешанного текста значительно выше, чем у исходного предложения. 
- Для правильного предложения на русском языке модель легко угадывает окончания, союзы и устойчивые словосочетания, поэтому её Loss мал, а следовательно, и PPL низкая.
- В перемешанном тексте отсутствует грамматика и логика, модель постоянно ошибается в предсказаниях, что ведет к высокому значению метрики. Это доказывает, что модель действительно выучила структуру русского языка